In [1]:
!pip install pymatgen matminer pymoo joblib pandas numpy

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.7/829.7 kB 20.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 95.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 48.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 100.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/879.5 kB 38.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 112.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 59.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 979.1/979.1 kB 19.4 MB/s  0:00:00
  Created wheel for bibtexparser: filename=bibtexparser-1.4.4-py3-none-any.whl size=43686 sha256=cbf93913d32f56b94e11f5e4fb2217aa38353610d08ca1b563e690212cf5ef07
  Stored in directory: /root/.cache/pip/wheels/43/ca/d8/0d315657fcf8ff8694a2f6701589bd1eaa602a8

In [2]:
import pandas as pd
import numpy as np
import joblib
from pymatgen.core import Composition
from matminer.featurizers.composition import ElementProperty
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize

# 1. Loading saved models
etr_tl = joblib.load('/etr_Tl_model.pkl')
etr_tx = joblib.load('/etr_Tx_model.pkl')
etr_tg = joblib.load('/etr_Tg_model.pkl')
ep_feat = ElementProperty.from_preset(preset_name="magpie")

# Defining feature lists globally so both the optimizer and saving loop can see them
features_tl = ['MagpieData mean MeltingT', 'MagpieData avg_dev Column', 'MagpieData avg_dev GSvolume_pa', 'MagpieData mode GSmagmom']
features_tx = ['MagpieData mean MeltingT', 'MagpieData avg_dev Column', 'MagpieData avg_dev GSvolume_pa', 'MagpieData mode GSmagmom']
features_tg = ['MagpieData mean MeltingT', 'MagpieData mean Column', 'MagpieData mean Electronegativity', 'MagpieData avg_dev GSvolume_pa']

class CompositionOptimization(ElementwiseProblem):
    def __init__(self):
        super().__init__(n_var=3, n_obj=2, n_constr=1, xl=np.array([5, 5, 5]), xu=np.array([20, 20, 10]))

    def _evaluate(self, x, out, *args, **kwargs):
        cu, ni, al = x
        zr = 100 - (cu + ni + al)

        if zr <= 0:
            out["G"] = [1.0]
            out["F"] = [[10000, 10000]]
            return

        out["G"] = [0.0]
        formula = f"Zr{zr:.1f}Cu{cu:.1f}Ni{ni:.1f}Al{al:.1f}"

        try:
            comp = Composition(formula)
            df_temp = pd.DataFrame({'Composition': [comp]})
            df_features = ep_feat.featurize_dataframe(df_temp, col_id="Composition", ignore_errors=True)

            # Predicting using the correct features for each model
            pred_tl = etr_tl.predict(df_features[features_tl])[0]
            pred_tx = etr_tx.predict(df_features[features_tx])[0]
            pred_tg = etr_tg.predict(df_features[features_tg])[0]

            obj1 = -pred_tx
            obj2 = - (1 / (pred_tl + pred_tg))

            out["F"] = [obj1, obj2]

        except Exception as e:
            out["F"] = [10000, 10000]

print("Starting NSGA-II Optimization... (This will take about 2 minutes)")
# Parameters set to 50 population, 20 generations
algorithm = NSGA2(pop_size=50)

res = minimize(CompositionOptimization(), algorithm, ('n_gen', 20), seed=42, verbose=True)

print("\nOptimization Complete! Saving Pareto Front...")
results_df = pd.DataFrame(res.X, columns=['Cu_wt%', 'Ni_wt%', 'Al_wt%'])
results_df['Zr_wt%'] = 100 - (results_df['Cu_wt%'] + results_df['Ni_wt%'] + results_df['Al_wt%'])

preds = []
for index, row in results_df.iterrows():
    formula = f"Zr{row['Zr_wt%']:.1f}Cu{row['Cu_wt%']:.1f}Ni{row['Ni_wt%']:.1f}Al{row['Al_wt%']:.1f}"
    comp = Composition(formula)
    df_temp = pd.DataFrame({'Composition': [comp]})
    df_features = ep_feat.featurize_dataframe(df_temp, col_id="Composition", ignore_errors=True)

    # Using the correct features again for the final saving loop
    tl = etr_tl.predict(df_features[features_tl])[0]
    tx = etr_tx.predict(df_features[features_tx])[0]
    tg = etr_tg.predict(df_features[features_tg])[0]
    preds.append({'Formula': formula, 'Pred_Tl': tl, 'Pred_Tx': tx, 'Pred_Tg': tg})

final_results = pd.concat([results_df, pd.DataFrame(preds)], axis=1)
# Saving results to csv
final_results.to_csv('nsga2_optimized_compositions.csv', index=False)

print("Results saved to 'nsga2_optimized_compositions.csv'")
print("\nTop 5 Optimized Alloys:")
print(final_results.head())

/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator ExtraTreeRegressor from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator ExtraTreesRegressor from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Starting NSGA-II Optimization... (This will take about 2 minutes)


ElementProperty:   0%|          | 0/1 [00:00<?, ?it/s]

ElementProperty:   0%|          | 0/1 [00:00<?, ?it/s]

ElementProperty:   0%|          | 0/1 [00:00<?, ?it/s]

ElementProperty:   0%|          | 0/1 [00:00<?, ?it/s]

... (1,063 outputs left out)

ElementProperty:   0%|          | 0/1 [00:00<?, ?it/s]

ElementProperty:   0%|          | 0/1 [00:00<?, ?it/s]

ElementProperty:   0%|          | 0/1 [00:00<?, ?it/s]

Results saved to 'nsga2_optimized_compositions.csv'

Top 5 Optimized Alloys:
      Cu_wt%     Ni_wt%    Al_wt%     Zr_wt%                   Formula  \
0  10.797102  17.549518  5.027275  66.626104   Zr66.6Cu10.8Ni17.5Al5.0   
1  19.858756  10.305271  9.878643  59.957330   Zr60.0Cu19.9Ni10.3Al9.9   
2  19.304704  12.425998  9.983167  58.286131  Zr58.3Cu19.3Ni12.4Al10.0   
3  19.964847  13.391480  9.891839  56.751834   Zr56.8Cu20.0Ni13.4Al9.9   
4  11.954367  18.079144  5.859549  64.106940   Zr64.1Cu12.0Ni18.1Al5.9   

    Pred_Tl  Pred_Tx  Pred_Tg  
0  1119.068  689.118  636.506  
1  1162.020  790.250  740.990  
2  1138.790  777.220  740.870  
3  1125.720  776.920  710.490  
4  1165.954  749.642  647.934  
